<a href="https://colab.research.google.com/github/IrisCheon/NLP-practice/blob/main/Womens_Clothing_Review_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Women's E-Commerce Clothing Reviews

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("nicapotato/womens-ecommerce-clothing-reviews")

print("Path to dataset files:", path)

100%|██████████| 2.79M/2.79M [00:00<00:00, 113MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/nicapotato/womens-ecommerce-clothing-reviews/versions/1


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

In [3]:
print(os.listdir(path))

['Womens Clothing E-Commerce Reviews.csv']


In [4]:
df = pd.read_csv(os.path.join(path, 'Womens Clothing E-Commerce Reviews.csv'))

#■ 데이터 확인

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23486 entries, 0 to 23485
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   Unnamed: 0               23486 non-null  int64 
 1   Clothing ID              23486 non-null  int64 
 2   Age                      23486 non-null  int64 
 3   Title                    19676 non-null  object
 4   Review Text              22641 non-null  object
 5   Rating                   23486 non-null  int64 
 6   Recommended IND          23486 non-null  int64 
 7   Positive Feedback Count  23486 non-null  int64 
 8   Division Name            23472 non-null  object
 9   Department Name          23472 non-null  object
 10  Class Name               23472 non-null  object
dtypes: int64(6), object(5)
memory usage: 2.0+ MB


In [8]:
df.shape

(23486, 11)

In [11]:
df.columns

Index(['Unnamed: 0', 'Clothing ID', 'Age', 'Title', 'Review Text', 'Rating',
       'Recommended IND', 'Positive Feedback Count', 'Division Name',
       'Department Name', 'Class Name'],
      dtype='object')

In [6]:
df.head()

,Unnamed: 0,Clothing ID,Age,Title,Review Text,Rating,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name
0,0,767,33,NaN,Absolutely wonderful - silky and sexy and comf...,4,1,0,Initmates,Intimate,Intimates
1,1,1080,34,NaN,Love this dress! it's sooo pretty. i happene...,5,1,4,General,Dresses,Dresses
2,2,1077,60,Some major design flaws,I had such high hopes for this dress and reall...,3,0,0,General,Dresses,Dresses
3,3,1049,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl...",5,1,0,General Petite,Bottoms,Pants
4,4,847,47,Flattering shirt,This shirt is very flattering to all due to th...,5,1,6,General,Tops,Blouses


In [12]:
df.isna().sum()

,0
Unnamed: 0,0
Clothing ID,0
Age,0
Title,3810
Review Text,845
Rating,0
Recommended IND,0
Positive Feedback Count,0
Division Name,14
Department Name,14


In [17]:
df = df.dropna(subset=["Review Text"]).copy()

# ■ 기본 EDA

In [20]:
df.columns

Index(['Unnamed: 0', 'Clothing ID', 'Age', 'Title', 'Review Text', 'Rating',
       'Recommended IND', 'Positive Feedback Count', 'Division Name',
       'Department Name', 'Class Name'],
      dtype='object')

In [21]:
df["Rating"].value_counts().sort_index()

,count
Rating,
1,821
2,1549
3,2823
4,4908
5,12540


In [22]:
df["Recommended IND"].value_counts()

,count
Recommended IND,
1,18540
0,4101


In [23]:
df["Department Name"].value_counts()

,count
Department Name,
Tops,10048
Dresses,6145
Bottoms,3662
Intimate,1653
Jackets,1002
Trend,118


In [24]:
df.groupby("Rating")["Recommended IND"].mean()

,Recommended IND
Rating,
1,0.018270
2,0.060684
3,0.414453
4,0.966585
5,0.998166


In [25]:
df["review_length"] = (
    df_clean["Review Text"].str.split().str.len()
)

In [26]:
# 추천여부별 단어 수

df.groupby("Recommended IND")["review_length"].mean()

,review_length
Recommended IND,
0,62.001951
1,59.797357


※ 비추천 리뷰에서 단어 수가 더 많지만, 큰 차이는 없음(3단어)

In [27]:
# 추천여부 x 상품 부문

department_summary = (
    df.groupby("Department Name").agg(
        review_count = ("Review Text", "size"),
        recommendation_rate = ("Recommended IND", "mean"),
        average_rating = ("Rating", "mean")
    ).sort_values("review_count", ascending=False)
)

department_summary

,review_count,recommendation_rate,average_rating
Department Name,,,
Tops,10048,0.810709,4.157743
Dresses,6145,0.805207,4.138812
Bottoms,3662,0.849536,4.278809
Intimate,1653,0.846340,4.271022
Jackets,1002,0.833333,4.254491
Trend,118,0.745763,3.838983


# ■ TF-IDF

In [28]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    stop_words = "english",
    min_df = 5,
    max_df = 0.8,
    max_features = 3000
)

tfidf_matrix = vectorizer.fit_transform(
    df["Review Text"]
)

feature_names = vectorizer.get_feature_names_out()

In [29]:
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns = feature_names,
    index = df_clean.index
)

# 추천 기준 tfidf 보기
recommendation_tfidf = (
    tfidf_df
    .assign(recommended = df["Recommended IND"])
    .groupby("recommended")
    .mean()
)

In [31]:
positive_terms = (
    recommendation_tfidf.loc[1].nlargest(15)
)

positive_terms

,1
dress,0.045567
love,0.043435
size,0.037290
great,0.035957
fit,0.031295
wear,0.031035
like,0.026808
perfect,0.024922
color,0.024329
just,0.023991


In [32]:
negative_terms = (
    recommendation_tfidf.loc[0].nlargest(15)
)

negative_terms

,0
dress,0.044409
like,0.041104
fabric,0.033683
just,0.031225
fit,0.029816
look,0.026929
size,0.025654
small,0.025485
really,0.024944
material,0.024739


In [35]:
term_difference = (
    recommendation_tfidf.loc[0]
    - recommendation_tfidf.loc[1]
)

nonrecommended_distinctive_terms = (
    term_difference.nlargest(15)
)

nonrecommended_distinctive_terms

,0
looked,0.017606
disappointed,0.015865
wanted,0.014703
like,0.014296
way,0.014097
returned,0.013688
unfortunately,0.012950
returning,0.012616
fabric,0.012603
huge,0.012451
